# 🚗 Entrenar: Reconocimiento de Marca y Modelo Vehicular
## Para usar con nuestro Vehicle Vision API

Este notebook entrena un clasificador de **marca y modelo** usando YOLOv8cls.

### Dataset: Stanford Cars desde HuggingFace (196 clases, ~16K imágenes, ~2GB)

**⚠️ Antes de empezar:**
- Entorno de ejecución → Cambiar tipo → **GPU** ✅
- **Ejecuta las celdas UNA POR UNA** en orden
- La celda 5 (entrenar) tarda ~1-2 horas

---

In [ ]:
# CELDA 1: Montar Google Drive + instalar dependencias
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics datasets -q

In [ ]:
# CELDA 2: Descargar Stanford Cars desde HuggingFace (sin autenticación)
from datasets import load_dataset

print("Descargando dataset Stanford Cars (~2GB)...")
dataset = load_dataset("tanganke/stanford_cars", split="train")
print(f"✅ {len(dataset)} imágenes de entrenamiento cargadas")

In [ ]:
# CELDA 3: Guardar imágenes de entrenamiento por clase
import os
from tqdm import tqdm

base = "/content/stanford_cars_yolo/train"
for i, item in enumerate(tqdm(dataset)):
    class_id = f"{item['label']:04d}"
    class_dir = f"{base}/{class_id}"
    os.makedirs(class_dir, exist_ok=True)
    item['image'].save(f"{class_dir}/img_{i:05d}.jpg")

print(f"✅ {len(dataset)} imágenes guardadas en {base}")

In [ ]:
# CELDA 4: Guardar imágenes de prueba/test
from datasets import load_dataset

test_dataset = load_dataset("tanganke/stanford_cars", split="test")

base_test = "/content/stanford_cars_yolo/test"
for i, item in enumerate(tqdm(test_dataset)):
    class_id = f"{item['label']:04d}"
    class_dir = f"{base_test}/{class_id}"
    os.makedirs(class_dir, exist_ok=True)
    item['image'].save(f"{class_dir}/img_{i:05d}.jpg")

print(f"✅ {len(test_dataset)} imágenes de prueba guardadas en {base_test}")

In [ ]:
# CELDA 5: ENTRENAR modelo YOLOv8cls (tarda ~1-2 horas con GPU)
from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')  # nano, ~6MB

results = model.train(
    data='/content/stanford_cars_yolo',
    epochs=50,
    imgsz=224,
    batch=32,
    device='cuda',
    workers=4,
    lr0=0.001,
    patience=10,
    name='marca_modelo',
    project='/content/drive/MyDrive/vehicle_ai'
)

print("✅ Entrenamiento completado")

In [ ]:
# CELDA 6: Exportar modelo a ONNX (para CPU en el VPS)
model.export(format='onnx', imgsz=224)

print("✅ Modelo exportado a ONNX")
print("📁 Ubicación:")
print("   /content/drive/MyDrive/vehicle_ai/marca_modelo/weights/best.onnx")
print()
print("📥 Para subir al VPS:")
print("   1. Descarga best.onnx desde tu Drive")
print("   2. En tu PC: scp best.onnx root@2.25.128.81:/opt/docker-projects/")
print("      plate_recognition/vehicle_vision/models/marca_modelo.onnx")